# 02 資料處理與視覺化 — 練習

用松柏護理之家退伍軍人症 line list（`legionella_outbreak.csv`）完成以下練習。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# Plotly: 確保在靜態建置（jupyter-book build）時也能輸出互動圖
pio.renderers.default = "notebook"


## 題目 1：讀入並檢視資料

1. 讀入 `data/synthetic/legionella_outbreak.csv`
2. 印出前 5 列和所有欄位名稱
3. 回答：有幾筆資料？幾個欄位？哪些欄位有遺漏值？

In [ ]:
# TODO: 讀入 CSV
# TODO: 印出 df.head() 和 df.columns.tolist()
# TODO: 用 df.info() 看遺漏值

## 題目 2：日期轉換與衍生變項

1. 把 `symptom_onset_date` 和 `hospitalization_date` 轉成 datetime
2. 建立 `infected` 欄位（`clinical_severity != 'not_ill'` → 1，否則 0）
3. 計算 `onset_to_hosp_days`（住院日期 − 發病日期）
4. 印出前 10 位感染者的 `case_id`、`symptom_onset_date`、`onset_to_hosp_days`

In [ ]:
# TODO: 日期轉換
# TODO: 建立 infected 欄位
# TODO: 計算 onset_to_hosp_days
# TODO: 篩選感染者，印出指定欄位

## 題目 3：流行曲線

用 matplotlib 畫一張流行曲線（epidemic curve）：
- 只取感染者
- X 軸：`symptom_onset_date`
- Y 軸：每日新增病例數
- 加上圖表標題和軸標籤

In [ ]:
# TODO: 篩選感染者
# TODO: groupby symptom_onset_date 計數
# TODO: 畫 bar chart
# TODO: 加標題、軸標籤

## 題目 4：翼區侵襲率比較圖

1. 用 `groupby(["floor", "wing"])` 計算每個翼區的住民數和感染人數
2. 計算侵襲率（%）
3. 用 seaborn 畫排序長條圖，在長條上方標上數字

In [ ]:
# TODO: groupby + agg
# TODO: 計算侵襲率
# TODO: 排序 + 畫 barplot
# TODO: 標數字

## 題目 5（挑戰題）：互動式分層流行曲線

用 Plotly 畫一張分層流行曲線：
- 按 `floor` 分色
- 使用堆疊長條圖（`barmode="stack"`）
- 加上中文標題和軸標籤

觀察：三個樓層的流行高峰是否同步？這代表什麼流行病學意義？

In [ ]:
# TODO: groupby symptom_onset_date + floor 計數
# TODO: 用 px.bar 畫互動式圖表

## 題目 6：頻率表與樞紐分析

1. 用 `value_counts()` 列出 `clinical_severity` 的次數分布（含百分比）
2. 用 `pd.pivot_table()` 做一張 **翼區 (wing) × 樓層 (floor)** 的侵襲率表格（提示：`values="infected"`, `aggfunc="mean"`）
3. 加上 `margins=True` 顯示小計

In [ ]:
# TODO: value_counts() 列出嚴重度分布
# TODO: value_counts(normalize=True) 加百分比
# TODO: pd.pivot_table() 建立翼區 × 樓層侵襲率表
# TODO: 加上 margins=True

## 題目 7：Method Chaining

用 method chaining（一行流水線）完成以下分析：
1. 篩選 70 歲以上的感染者（提示：`.query("infected == 1 and age >= 70")`）
2. 按 `floor` 分組
3. 用 `.agg()` 同時算人數和死亡數
4. 新增致死率欄位（`.assign(cfr=lambda d: ...)`）
5. 按致死率由高到低排序

In [ ]:
# TODO: 用 method chaining 完成上述分析
# 提示：(df.query(...).groupby(...).agg(...).assign(...).sort_values(...))

## 題目 8：合併資料與文字清理

1. 模擬一份實驗室檢驗 DataFrame（`lab`），包含 `case_id` 和 `ct_value`
2. 用 `pd.merge()` 把 `lab` 左連接到 `df`
3. 用 `.str.strip().str.upper()` 統一 `wing` 欄位大小寫
4. 用 `drop_duplicates("case_id")` 去除重複通報
5. 用 `nlargest(5, "age")` 找出年齡最大的 5 位感染者

In [ ]:
# TODO: 建立 lab DataFrame（case_id, ct_value）
# TODO: pd.merge(df, lab, on="case_id", how="left")
# TODO: str.strip().str.upper() 統一 wing
# TODO: drop_duplicates("case_id")
# TODO: nlargest(5, "age")